<a href="https://colab.research.google.com/github/miaflynn/CYPLAN255-Final-Project/blob/main/03d_visualizations_survival_rate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Analysis still in progress

##Importing

In [ ]:

import pandas as pd
import geopandas as gpd
import numpy as np
import os
%matplotlib inline
import matplotlib.pyplot as plt
from shapely.geometry import LineString
import plotly.express as px



from google.colab import drive
drive.mount('/content/drive')

!wget https://raw.githubusercontent.com/miaflynn/CYPLAN255-Final-Project/main/functions.py
import functions as fx



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Saving functions.py to functions.py


{'functions.py': b'import pandas as pd\nimport numpy as np\nimport geopandas as gpd\n\n\ndef group_points_by_poly_year (points: gpd.GeoDataFrame, polygons: gpd.GeoDataFrame):\n    """\n    Groups all the business location points by tract, year and status (open or closed)\n    \n    Parameters:\n        points: geodataframe with point data\n        polygons: geodataframe with tract geometries\n    \n    Returns:\n        GeoDataFrame\n    """\n\n    points = gpd.sjoin(points, polygons, how="left", predicate="within")\n\n    year_col = \'year_open\' if \'year_open\' in points.columns else \'year\'\n\n    tract_year = (\n        points\n        .groupby(["GEOID", year_col, "status"])\n        .size()\n        .reset_index(name="count")\n        .pivot(index=["GEOID", year_col], columns="status", values="count")\n        .fillna(0)\n        .reset_index()\n        .sort_values(year_col)\n    )\n\n    tracts_plot = polygons[["GEOID", "geometry"]].merge(\n        tract_year,\n        on="GEO

In [74]:
import functions
import importlib
importlib.reload(functions)

<module 'functions' from '/content/functions.py'>

##Reading in file

In [75]:
gdf = gpd.read_file("/content/drive/MyDrive/C255_final_project/cleaned/rde_biz_all_startdate.geojson")

##Reading in Census Tracts

In [76]:
sf_tracts = gpd.read_file("/content/drive/MyDrive/C255_final_project/cleaned/sf_tracts.geojson")

In [85]:
sf_block_grp = gpd.read_file("/content/drive/MyDrive/C255_final_project/cleaned/sf_block_grp.geojson")
sf_block_grp = sf_block_grp.rename(columns={'geoid': 'GEOID'})

In [82]:
gdf['location_end_date'] = pd.to_datetime(gdf['location_end_date'])
gdf['location_start_date'] = pd.to_datetime(gdf['location_start_date'])
gdf['open_month_year'] = gdf['location_start_date'].dt.strftime('%B %Y')
gdf['close_month_year'] = gdf['location_end_date'].dt.strftime('%B %Y')

In [83]:
pre_covid_mask = gdf['location_start_date'] < '2020-03-01'
pre_covid_tracts = gdf[pre_covid_mask]

survival_mask = pre_covid_tracts['location_end_date'].isna()
survival_tracts = pre_covid_tracts[survival_mask].sort_values('location_start_date')

In [88]:
pre_covid_tracts_grouped = functions.group_points_by_poly(
    points=pre_covid_tracts,
    polygons=sf_tracts
)

survival_tracts_grouped = functions.group_points_by_poly(
    points=survival_tracts,
    polygons=sf_tracts
)


# adding closed businesses and opened businesses that have not closed
# (because closed businesses also are listed in opened, just in a diff year)
survival_tracts_grouped['total'] = pre_covid_tracts_grouped['closed'] + survival_tracts_grouped['opened']
survival_tracts_grouped['survival_rate'] = survival_tracts_grouped['opened'] / survival_tracts_grouped['total']

survival_tracts_grouped

,GEOID,geometry,opened,total,survival_rate
0,06075980401,"POLYGON ((-123.00779 37.69894, -123.00755 37.7...",0.0,0.0,NaN
1,06075060400,"POLYGON ((-122.5068 37.73556, -122.50614 37.73...",9.0,14.0,0.642857
2,06075035400,"POLYGON ((-122.50846 37.74892, -122.50794 37.7...",34.0,70.0,0.485714
3,06075035300,"POLYGON ((-122.49978 37.74947, -122.49991 37.7...",23.0,64.0,0.359375
4,06075032901,"POLYGON ((-122.48586 37.75008, -122.48573 37.7...",16.0,48.0,0.333333
...,...,...,...,...,...
246,06075012602,"POLYGON ((-122.44103 37.80669, -122.43564 37.8...",37.0,72.0,0.513889
247,06075010202,"POLYGON ((-122.42152 37.80844, -122.42026 37.8...",66.0,123.0,0.536585
248,06075010101,"POLYGON ((-122.41586 37.80869, -122.41266 37.8...",90.0,221.0,0.407240
249,06041131100,"MULTIPOLYGON (((-122.47807 37.83272, -122.4781...",0.0,0.0,NaN


In [89]:
import plotly.graph_objects as go

fig = px.choropleth_mapbox(
    survival_tracts_grouped,
    geojson=survival_tracts_grouped.set_index("GEOID").__geo_interface__,
    locations="GEOID",
    color="survival_rate",
    hover_name="GEOID",
    center={"lat": 37.7749, "lon": -122.4194},
    zoom=10,
    mapbox_style="carto-positron",
    color_continuous_scale="Reds",
    height=500,
    width=700

)


fig.show()

In [99]:
fig = px.density_mapbox(
    survival_tracts,
    lat=survival_tracts.geometry.y,
    lon=survival_tracts.geometry.x,
    radius=3,
    center={"lat": 37.7749, "lon": -122.4194},
    zoom=11,
    mapbox_style="carto-positron",
    height=600
)
fig.show()